# Table de modélisation : audit, métadonnées et redondances

Contributeurs : Cécile · Date : 31/10/2026

Table produite à la fin de feature_engineering (table_modelisation_bv.parquet) : une ligne par année × sous-bassin × type de bien, seulement les lignes où le prix et le prix N-1 sont renseignés. Trois objectifs :

- un fichier de métadonnées qui décrit chaque variable ;
- une seule variable par information (par exemple le taux plutôt que le nombre de faits) ;
- les corrélations entre variables, pour repérer les informations redondantes.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# config.py se trouve à la racine du projet : on remonte les dossiers jusqu'à le trouver
racine = Path.cwd()
while not (racine / "config.py").exists() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
from config import DATA_RAW, DATA_OUT

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 1. Audit : contenu de la table

On charge la table et on regarde, pour chaque variable, son type, sa part de valeurs vides et son nombre de valeurs distinctes. Une variable qui n'a qu'une seule valeur, ou très peu, se repère tout de suite.

In [ ]:
# Table lue par commun.py : index annee x sous_bassin_de_vie x type_bien
table = pd.read_parquet(DATA_OUT / "table_modelisation_bv.parquet")
print(table.shape)
print("Index :", table.index.names, "| unique :", table.index.is_unique)

# Lignes par année et par type de bien
display(table.groupby(level=["annee", "type_bien"]).size().unstack())

# Une ligne par variable : type, part de valeurs vides, nombre de valeurs distinctes
apercu = pd.DataFrame({
    "type": table.dtypes.astype(str),
    "% vides": table.isna().mean().mul(100).round(1),
    "valeurs distinctes": table.nunique(),
})
display(apercu)

**Lecture**

- **28 450 lignes** (8 891 appartements, 19 559 maisons), années de prix 2015 à 2025, 38 colonnes. Index unique.
- **Une seule variable par information** : délinquance en taux lissés sur 3 ans, équipements en taux pour 10 000 habitants, densité de population, Filosofi contrôlé par la part des ménages couverts. Les nombres bruts ont été retirés dans le feature engineering.
- **Valeurs vides : seulement la délinquance, pour les prix 2015 et 2016** (17,4 % des lignes) : les faits commencent en 2016 et sont décalés d'un an. À partir de 2017, aucune valeur ne manque.
- **Photos fixes** : logement (2022), LOVAC (2023), Filosofi (2021), équipements (2025) et densité (2024) ont une valeur par sous-bassin, identique toutes les années (environ 1 810 valeurs distinctes). La densité n'en a que 1 772 : les arrondissements d'une même ville ont la densité de leur ville.
- **Nombre restant** : `nb_ventes_n1` (ventes de N-1) suit la taille du sous-bassin ; son évolution entre N-2 et N-1 reste à décider. `n_ventes` (ventes de N) ne sert qu'au seuil de 20 ventes, jamais comme variable explicative.


## 2. Nature des variables : nombre ou taux, et base de calcul

Pour chaque variable : sa source, sa nature (nombre, taux, part, moyenne, prix) et sa base de calcul, relues dans le code du feature engineering. Un nombre suit d'abord la taille du sous-bassin ; un taux ou une part permet de comparer des territoires de tailles différentes.

In [ ]:
# Pour chaque variable : (source, nature, base de calcul)
nature = {}

# DVF+ : année, type de bien, ventes, prix
nature["annee"]        = ("DVF+", "année", "année des prix (N)")
nature["maison"]       = ("DVF+", "indicatrice", "1 = maison, 0 = appartement")
nature["n_ventes"]     = ("DVF+", "nombre", "ventes de l'année N (sert seulement au seuil de 20 ventes)")
nature["nb_ventes_n1"] = ("DVF+", "nombre", "ventes de l'année N-1, additionnées au sous-bassin")
nature["prix"]         = ("DVF+", "prix (€/m²)", "médiane des prix au m² des ventes de N")
nature["prix_n1"]      = ("DVF+", "prix (€/m²)", "médiane des prix au m² des ventes de N-1")

# Délinquance (SSMSI) : faits et dénominateurs additionnés au sous-bassin, puis taux lissé sur 3 ans
FAMILLES = ["vols_vehicules", "vols_violents", "vols_sans_violence", "cambriolages",
            "destructions", "stupefiants", "violences_intrafamiliales", "violences_sexuelles"]
for f in FAMILLES:
    denom = "1 000 logements" if f == "cambriolages" else "1 000 habitants"
    nature["taux3_" + f] = ("Délinquance", "taux",
        f"faits de N-3 à N-1 pour {denom} des mêmes années (1 an pour les prix 2017, 2 ans pour 2018)")
nature["part_indicateurs_estimes"] = ("Délinquance", "part (%)",
    "indicateurs estimés (secret statistique) sur 15, moyenne pondérée par la population")

# LOVAC (photo 2023)
nature["taux_vacance_complete"] = ("LOVAC", "taux (%)",
    "taux communal (complété par l'EPCI si masqué), moyenne pondérée par les logements du recensement (P22_LOG)")
nature["part_logements_taux_impute"] = ("LOVAC", "part (%)",
    "logements situés dans une commune dont le taux a été complété par l'EPCI / P22_LOG")

# Équipements (BPE 2025, photo) : équipements et habitants 2024 additionnés au sous-bassin
for d in ["commerces", "sante", "services", "sports", "tourisme", "transports"]:
    nature["taux_equip_" + d] = ("Équipements", "taux", "équipements du domaine pour 10 000 habitants (2024)")

# Densité (photo) : habitants 2024 et superficie additionnés au sous-bassin
nature["densite_population"] = ("Densité", "taux", "habitants 2024 / km² (arrondissements : densité de leur ville)")

# Parc de logements (recensement 2022, photo) : effectifs additionnés au sous-bassin, puis formule
LOGEMENT = {
    "taux_vacance":       ("part (%)", "P22_LOGVAC / P22_LOG"),
    "part_rsec":          ("part (%)", "P22_RSECOCC / P22_LOG"),
    "part_apparts":       ("part (%)", "P22_APPART / P22_LOG"),
    "log_nb_logements":   ("log d'un nombre", "log10(P22_LOG)"),
    "part_moins40m2":     ("part (%)", "(P22_RP_M30M2 + P22_RP_3040M2) / P22_RP"),
    "part_avant1946":     ("part (%)", "(P22_RP_ACH1919 + P22_RP_ACH1945) / P22_RP"),
    "part_2006_2019":     ("part (%)", "P22_RP_ACH2019 / P22_RP"),
    "anciennete_moyenne": ("moyenne", "P22_ANEM_RP / P22_RP"),
    "taille_menage":      ("moyenne", "P22_NPER_RP / P22_RP"),
    "part_sans_voiture":  ("part (%)", "100 - P22_RP_VOIT1P / P22_MEN"),
    "part_parking":       ("part (%)", "P22_RP_GARL / P22_MEN"),
    "part_suroccupes":    ("part (%)", "(C22_RP_SUROCC_MOD + C22_RP_SUROCC_ACC) / P22_RP"),
}
for var, (nat, formule) in LOGEMENT.items():
    nature[var] = ("Logement", nat, formule)

# Filosofi (photo 2021)
nature["niveau_vie_median"] = ("Filosofi", "moyenne (€)",
    "médianes communales, moyenne pondérée par les ménages fiscaux (communes à médiane connue)")
nature["part_menages_couverts"] = ("Filosofi", "part (%)",
    "ménages fiscaux des communes à médiane connue / ménages du recensement 2022 (P22_MEN)")

# Contrôle : toutes les variables de la table sont décrites, et rien de plus
assert set(nature) == set(table.columns), set(nature) ^ set(table.columns)

nature_vars = pd.DataFrame.from_dict(nature, orient="index", columns=["source", "nature", "base de calcul"])
display(nature_vars)
display(pd.crosstab(nature_vars["source"], nature_vars["nature"]))

**Lecture**

- **Délinquance, équipements, densité, LOVAC, logement** : uniquement des taux, des parts ou des moyennes, calculés à partir d'effectifs additionnés au sous-bassin. Ils comparent des territoires de tailles différentes.
- **Filosofi** : une moyenne (niveau de vie) et une part de contrôle (ménages couverts).
- **Deux variables de contrôle qualité**, à ne pas confondre avec des variables explicatives : `part_indicateurs_estimes` (délinquance) et `part_logements_taux_impute` (LOVAC), auxquelles s'ajoute `part_menages_couverts` (Filosofi).
- **Un seul nombre reste parmi les variables explicatives** : `nb_ventes_n1`, qui suit la taille du sous-bassin. `n_ventes` ne sert qu'au seuil de 20 ventes.
- La taille du territoire est portée par une variable dédiée : `log_nb_logements`.

## 3. Distributions et valeurs extrêmes

Sur toutes les lignes de la table (2015-2025), telles que le modèle les verra. Pour chaque variable : quelques centiles, le rapport entre le maximum et le 99e centile, et l'asymétrie. Les variables très asymétriques sont à transformer avant la modélisation (par exemple en logarithme).

**Règles de lecture :**
- **max / p99** : proche de 1, le maximum reste dans la continuité des autres valeurs ; nettement au-dessus de 1, quelques valeurs isolées tirent la distribution.
- **Asymétrie** : autour de 0, distribution symétrique ; au-delà de 1, longue queue vers les grandes valeurs ; en dessous de −1, longue queue vers les petites valeurs.

In [ ]:
# Toutes les variables numériques, sauf l'année et l'indicatrice maison
variables = [c for c in table.columns if c not in ["annee", "maison"]]

# 1. Centiles, valeurs extrêmes et asymétrie
quantiles = table[variables].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T
quantiles["max / p99"] = quantiles["max"] / quantiles["99%"]
quantiles["asymetrie"] = table[variables].skew()
display(quantiles[["min", "1%", "5%", "50%", "95%", "99%", "max", "max / p99", "asymetrie"]].round(2))

# 2. Histogrammes des 36 variables (grille 9 x 4)
fig, axes = plt.subplots(9, 4, figsize=(20, 36))
axes = axes.flatten()
for ax, col in zip(axes, variables):
    sns.histplot(table[col].dropna(), bins=50, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
for ax in axes[len(variables):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()


**Lecture**

- **Distributions proches de la symétrie** (asymétrie entre −1 et 1) : taux de vacance (recensement et LOVAC), ancienneté d'emménagement, taille des ménages, logements anciens et récents, `log_nb_logements`. Pas de transformation nécessaire.
- **Longue queue vers les grandes valeurs** (asymétrie > 1), candidates à une transformation logarithmique avant une régression linéaire :
  - le prix et le prix N-1 (3,45 et 3,51) : la plupart des sous-bassins sont entre 900 et 4 300 €/m², Paris monte à 19 652 €/m² ;
  - les ventes de N-1 (33,77) : médiane de 148 ventes, maximum de 70 042 ;
  - la densité (8,31) : médiane de 91 habitants/km², environ 20 000 pour les arrondissements de Paris (le pic isolé à droite de l'histogramme) ;
  - plusieurs taux de délinquance (vols sans violence 19,40 ; vols violents 14,23 ; stupéfiants 13,63 ; vols de véhicules 5,04) ;
  - plusieurs taux d'équipements (transports 14,65 ; commerces 6,64 ; tourisme 5,50 ; services 5,22) ;
  - logement et revenus : résidences secondaires, petits logements, ménages sans voiture, suroccupation, niveau de vie.
- **Valeurs isolées** (max / p99 très au-dessus de 1) : ventes de N-1 (28,1), vols sans violence (11,6 : jusqu'à 570 faits pour 1 000 habitants), stupéfiants (12,4), vols violents (8,3), équipements de transport (8,3). Ce sont des territoires où beaucoup de faits ou d'équipements se rapportent à peu d'habitants (centres touristiques, quartiers d'affaires) : des valeurs réelles, pas des erreurs.
- **Des zéros** dans plusieurs variables asymétriques (vols violents, stupéfiants, équipements de tourisme et de transport) : une transformation en log(1 + x) plutôt qu'en log(x).
- **Contrôles qualité :**
  - `part_indicateurs_estimes` : médiane de 48 %. Dans la moitié des sous-bassins, près de la moitié des indicateurs de délinquance sont estimés ; le pic à 0 correspond aux grands territoires, sans secret statistique.
  - `part_menages_couverts` : médiane de 96 %, mais 5 % des lignes dépassent 101 % (jusqu'à 119 %) : les ménages fiscaux et ceux du recensement ne sont pas définis de la même façon.

### À prendre en compte pour la modélisation : transformation logarithmique

**log** = log(x) · **log1p** = log(1 + x), pour les variables qui contiennent des zéros · **—** = pas de transformation

| Groupe (nombre de variables) | Variables | Régression linéaire | Modèle hiérarchique | Arbre, forêt, boosting | Réseau de neurones |
|---|---|---|---|---|---|
| Cible (1) | `prix` | log au choix | log au choix | log au choix | log au choix |
| Prix N-1 (1) | `prix_n1` | log si cible en log | log si cible en log | — | log si cible en log |
| Ventes N-1 (1) | `nb_ventes_n1` | log | log | — | log |
| Délinquance (8) | `taux3_` : vols de véhicules, vols violents, vols sans violence, cambriolages, destructions, stupéfiants, violences intrafamiliales, violences sexuelles | log (log1p : vols violents, stupéfiants) | idem | — | idem |
| Équipements (6) | `taux_equip_` : commerces, santé, services, sports, tourisme, transports | log (log1p : tourisme, transports) | idem | — | idem |
| Densité (1) | `densite_population` | log | log | — | log |
| Revenus (1) | `niveau_vie_median` | log | log | — | log |
| Logement, asymétriques (5) | `part_sans_voiture`, `part_moins40m2`, `part_suroccupes`, `part_rsec`, `part_apparts` | log | log | — | log |
| Logement, symétriques (7) | `taux_vacance`, `part_avant1946`, `part_2006_2019`, `anciennete_moyenne`, `taille_menage`, `part_parking`, `log_nb_logements` | — | — | — | — |
| Vacance LOVAC (1) | `taux_vacance_complete` | — | — | — | — |
| Contrôle qualité (3) | `part_indicateurs_estimes`, `part_logements_taux_impute`, `part_menages_couverts` | — | — | — | — |
| Type de bien (1) | `maison` | — | — | — | — |
| Hors modèle (2) | `annee`, `n_ventes` | — | — | — | — |

Total : 38 variables. Cambriolages et violences intrafamiliales sont juste au-dessus du seuil d'asymétrie (1,19 et 1,12) : leur log est facultatif. Réseau de neurones : toutes les variables sont ensuite mises à l'échelle (paramètres calculés sur les années d'apprentissage seulement). La même table peut servir à tous les modèles : le log ne change pas les coupures d'un arbre.

**⚠️ Interprétation**

| Résultat | Variable en log, cible en euros | Variable et cible en log |
|---|---|---|
| Coefficient de régression | +1 % de la variable → coefficient / 100 €/m² | +1 % de la variable → +coefficient % du prix |
| Seuil d'un arbre | seuil en log : le retransformer (exponentielle) pour le lire | idem |
| Valeur SHAP | en €/m², s'additionnent jusqu'au prix | en log : +0,05 ≈ **+5 % du prix**, jamais +0,05 €/m² ; les contributions se multiplient sur le prix en euros |
| Prévision | en €/m² | retransformer (exponentielle) avant de calculer l'erreur ; léger biais vers le bas |

### Transformation logarithmique des variables asymétriques

Variables identifiées en section 3 (asymétrie > 1). Logarithme naturel : log(x) pour les variables toujours positives, log(1 + x) pour celles qui contiennent des zéros. Transformation faite sur une copie de la table, pour l'exploration : elle n'est pas exportée. On vérifie que l'asymétrie est corrigée.

In [ ]:
# Variables toujours > 0 : log(x)
LOG = ["nb_ventes_n1", "densite_population", "niveau_vie_median",
       "taux3_vols_sans_violence", "taux3_vols_vehicules", "taux3_destructions",
       "taux3_violences_sexuelles", "taux3_cambriolages", "taux3_violences_intrafamiliales",
       "taux_equip_commerces", "taux_equip_services", "taux_equip_sante", "taux_equip_sports",
       "part_sans_voiture", "part_moins40m2", "part_suroccupes", "part_rsec", "part_apparts"]
# Variables avec des zéros : log(1 + x)
LOG1P = ["taux3_vols_violents", "taux3_stupefiants", "taux_equip_tourisme", "taux_equip_transports"]

# Contrôle : aucune valeur <= 0 dans les variables passées en log(x)
assert (table[LOG] > 0).all().all() or table[LOG].min().min() > 0, "Valeur nulle ou négative dans LOG"

table_log = table.copy()
table_log[LOG] = np.log(table_log[LOG])
table_log[LOG1P] = np.log1p(table_log[LOG1P])

# Asymétrie avant et après
asym = pd.DataFrame({"avant": table[LOG + LOG1P].skew(),
                     "après": table_log[LOG + LOG1P].skew()}).round(2)
display(asym.sort_values("avant", ascending=False))

## 4. Redondances entre variables

**Pourquoi regarder cela :** deux variables qui portent la même information n'apportent rien de plus au modèle ; en régression linéaire, elles rendent même les coefficients instables. On n'en garde alors qu'une.

**Méthode :** corrélation de Spearman entre toutes les variables explicatives, sur toutes les lignes de la table. Spearman compare les classements : il n'est pas sensible aux valeurs extrêmes ni à une transformation en log. Seuil de redondance : |rho| ≥ 0,8, comme dans l'EDA (rapport, section 4.8 ; EDA complémentaire, B1).

Hors calcul : la cible (`prix`), les variables hors modèle (`annee`, `n_ventes`) et l'indicatrice `maison`.

On regarde aussi les couples signalés comme « à contrôler » dans le feature engineering (sections 7).

In [ ]:
# Variables explicatives : tout sauf la cible, les variables hors modèle et l'indicatrice maison
explicatives = [c for c in table.columns if c not in ["prix", "annee", "n_ventes", "maison"]]
rho = table[explicatives].corr(method="spearman")

# 1. Couples redondants : |rho| >= 0,8, du plus fort au moins fort (chaque couple une seule fois)
paires = rho.where(np.triu(np.ones(rho.shape, dtype=bool), k=1)).stack().rename("rho").reset_index()
paires.columns = ["variable 1", "variable 2", "rho"]
paires = paires.reindex(paires["rho"].abs().sort_values(ascending=False).index)
print("Couples à |rho| >= 0,8 :")
display(paires[paires["rho"].abs() >= 0.8].round(2))

# 2. Couples à contrôler, signalés dans le feature engineering
A_CONTROLER = [
    ("taux3_vols_sans_violence", "taux3_destructions"),
    ("taux_vacance", "taux_vacance_complete"),
    ("part_moins40m2", "part_suroccupes"),
    ("part_sans_voiture", "part_parking"),
    ("densite_population", "log_nb_logements"),
    ("densite_population", "part_apparts"),
]
print("Couples à contrôler :")
display(pd.DataFrame([(a, b, round(rho.loc[a, b], 2)) for a, b in A_CONTROLER],
                     columns=["variable 1", "variable 2", "rho"]))

**Comment lire les tableaux (exemple : petits logements et logements suroccupés, rho = 0,88) :**
- **Ce que mesure rho :** on classe les sous-bassins selon leur part de logements de moins de 40 m², puis selon leur part de logements suroccupés, et on compare les deux classements.
- **Le signe :** positif, les deux variables montent ensemble ; négatif, l'une monte quand l'autre baisse (ménages sans voiture et logements avec parking, rho = −0,84).
- **La valeur :** 0,88 veut dire que les deux classements sont presque les mêmes. Repères : 0,1 lien faible, 0,3 moyen, 0,5 fort ; au-delà de 0,8, les deux variables portent presque la même information.

**Constat :**
- **Deux couples redondants** (|rho| ≥ 0,8), tous les deux dans le parc de logements : petits logements / logements suroccupés (0,88) et ménages sans voiture / logements avec parking (−0,84). Il faut n'en garder qu'une variable par couple.
- **Couples à contrôler, sous le seuil :** vols sans violence / destructions (0,78) restent juste en dessous : on garde les deux. Vacance du recensement / vacance LOVAC (0,63) : liées, mais elles ne mesurent pas la même chose (tout le parc ou le parc privé). Densité / nombre de logements et densité / part d'appartements (0,46) : pas de redondance.
- Aucune redondance entre sources différentes : chaque source apporte sa propre information.

## 5. Importance des variables

Pour voir rapidement quelles variables comptent le plus pour prévoir le prix : deux modèles simples (régression linéaire, forêt aléatoire), un par type de bien, lus avec SHAP. Ce n'est pas une évaluation de modèle : sans le prix N-1, réglages non optimisés.

### 5.1 Données d'apprentissage

Table transformée en log (section 3), prix de 2017 à 2023 (ni la validation 2024, ni le test 2025). Les lignes avec une case vide sont retirées, la régression linéaire ne les accepte pas.

In [ ]:
import shap
from sklearn.linear_model import LinearRegression

AVEC_PRIX_N1 = False   # True : prix N-1 inclus (il domine tout)

variables = [c for c in explicatives if AVEC_PRIX_N1 or c != "prix_n1"] + ["maison"]
apprentissage = table_log[(table_log["annee"] >= 2017) & (table_log["annee"] <= 2023)]

# Diagnostic : cases vides restantes de 2017 à 2023, par variable et par année
vides = apprentissage[variables].isna()
print("Variables avec des cases vides :")
print(vides.sum()[vides.sum() > 0].to_string())
print("Lignes concernées par année :")
print(vides.any(axis=1).groupby(apprentissage["annee"]).sum().to_string())

# La régression n'accepte pas les cases vides : lignes retirées
apprentissage = apprentissage.dropna(subset=variables)
print("Lignes gardées :", len(apprentissage))

**Lecture :** seules `taux_vacance_complete` et `part_logements_taux_impute` (LOVAC) ont des cases vides : une ligne par année, un sous-bassin sans donnée LOVAC. 18 286 lignes gardées.

### 5.2 SHAP par type de bien (régression linéaire, toutes les variables)

Une régression linéaire par type de bien (maisons, appartements), mêmes réglages que ci-dessus : table transformée en log, prix de 2017 à 2023, lignes sans donnée LOVAC retirées, cible en €/m². On compare quelles variables comptent pour chaque marché.

In [ ]:
variables_type = [c for c in explicatives if AVEC_PRIX_N1 or c != "prix_n1"]   # sans "maison"

plt.figure(figsize=(22, 10))
for i, (nom, valeur) in enumerate([("Maisons", 1), ("Appartements", 0)], start=1):
    app_type = apprentissage[apprentissage["maison"] == valeur]
    X_t, y_t = app_type[variables_type], app_type["prix"]
    modele_t = LinearRegression().fit(X_t, y_t)

    echantillon_t = X_t.sample(min(2000, len(X_t)), random_state=42)
    valeurs_t = shap.LinearExplainer(modele_t, X_t).shap_values(echantillon_t)
    print(nom, ":", len(X_t), "lignes d'apprentissage")

    # Chaque graphique se dessine dans sa moitié de la figure
    plt.subplot(1, 2, i)
    shap.summary_plot(valeurs_t, echantillon_t, max_display=20, show=False,
                      plot_size=None, color_bar=(i == 2))
    plt.title(f"SHAP – {nom} (régression linéaire, 2017-2023)")

plt.tight_layout()
plt.show()

**Comment lire les graphiques :** chaque point est un sous-bassin × année. Sa position dit de combien la variable fait monter (à droite) ou baisser (à gauche) le prix prévu, en €/m², par rapport au prix moyen. Sa couleur donne la valeur de la variable : rouge = élevée, bleu = faible. Les variables sont classées de la plus utilisée à la moins utilisée.
*Exemple : pour les appartements, un niveau de vie élevé (points rouges) ajoute jusqu'à +4 500 €/m² au prix prévu ; un niveau de vie faible (points bleus) retire jusqu'à −3 500 €/m².*

**Ce qui est commun aux deux marchés**
- **Le niveau de vie est la variable la plus forte**, pour les maisons (de −2 000 à +4 000 €/m²) comme pour les appartements (de −3 500 à +4 500 €/m²).
- **Même sens pour plusieurs variables** : une forte vacance fait baisser le prix ; beaucoup de logements récents (2006-2019) et de résidences secondaires le font monter ; beaucoup d'équipements de sport (signe d'un territoire rural) le font baisser.

**Ce qui distingue les deux marchés**
- **Maisons : le cadre de vie du territoire.** Après le niveau de vie viennent les équipements de services et la densité. Le prix des maisons suit ce qu'offre le territoire autour.
- **Appartements : la centralité urbaine.** Après le niveau de vie viennent les ménages sans voiture, les vols sans violence, les logements avec parking (en négatif), puis la densité, les petits logements, les logements d'avant 1946 et les équipements de tourisme. Ce sont des marqueurs de centres-villes anciens et touristiques, où les appartements sont les plus chers.
- **Ventes de l'année précédente** : 4e variable pour les maisons (beaucoup de ventes, prix plus bas), absente des 20 premières pour les appartements.

**Limites : ce que ces graphiques ne permettent pas de conclure**
- **Ce n'est pas une relation de cause à effet** : les vols ne font pas monter les prix ; ils signalent des centres-villes, où les prix sont élevés.
- **Les variables liées entre elles se partagent ou s'opposent leurs contributions.** Leurs signes ne sont pas à interpréter séparément :
  - ménages sans voiture / parking (rho = −0,84), tous deux dans les 4 premières variables pour les appartements ;
  - vols sans violence / destructions (rho = 0,78), qui partent en sens opposés ;
  - violences intrafamiliales / violences sexuelles (rho = 0,7) ;
  - ventes de N-1 / nombre de logements (rho = 0,6) ;
  - la part d'appartements, qui fait baisser le prix dans les deux modèles, alors qu'elle est liée positivement à la densité et aux petits logements.
- **Une variable de contrôle qualité est utilisée par le modèle des maisons** : `part_logements_taux_impute` (taux de vacance complété par l'intercommunalité) y sert d'indice du caractère rural.
- **Régression linéaire simple**, sans le prix N-1, sur les prix 2017-2023 : 12 453 lignes pour les maisons, 5 833 pour les appartements. Ce n'est pas une évaluation de modèle.

**Conséquences pour la modélisation**
- Les marchés n'utilisent pas les mêmes variables : cela confirme le choix de modéliser maisons et appartements séparément.
- Les deux couples redondants sont à réduire à une variable, en priorité ménages sans voiture / parking, qui pèse le plus pour les appartements.
- Les variables de contrôle qualité sont à décider : les retirer du modèle, ou les garder en sachant qu'elles mesurent en partie le caractère rural.

### 5.3 Effet du retrait des variables redondantes

Une variable retirée par couple redondant : `part_suroccupes` (gardée : `part_moins40m2`), `part_parking` (gardée : `part_sans_voiture`), `taux3_destructions` (gardée : `taux3_vols_sans_violence`). Critère : lien le plus fort et le plus stable avec le prix, pour les deux types de bien.

**Évaluation :** régression linéaire par type de bien, entraînée sur les prix de 2017 à 2023, mesurée sur 2024 (année de validation ; 2025 reste réservée au test). Deux mesures :
- **R²** : part des écarts de prix entre sous-bassins que le modèle explique (1 = tout, 0 = rien) ;
- **MAE** : erreur moyenne en €/m², en valeur absolue.

Sans le prix N-1, l'erreur reste élevée : l'objectif est de comparer les deux jeux de variables, pas d'évaluer un modèle final.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

RETIREES = ["part_suroccupes", "part_parking", "taux3_destructions"]
JEUX = {"toutes les variables": variables_type,
        "sans redondances": [c for c in variables_type if c not in RETIREES]}

# Validation : prix 2024, lignes sans donnée LOVAC retirées
validation = table_log[table_log["annee"] == 2024].dropna(subset=variables_type)

resultats, modeles = [], {}
for nom_jeu, vars_jeu in JEUX.items():
    for nom, valeur in [("Maisons", 1), ("Appartements", 0)]:
        app_t = apprentissage[apprentissage["maison"] == valeur]
        val_t = validation[validation["maison"] == valeur]
        modele_t = LinearRegression().fit(app_t[vars_jeu], app_t["prix"])
        modeles[(nom_jeu, nom)] = (modele_t, app_t[vars_jeu])
        resultats.append({
            "variables": nom_jeu, "type": nom, "nb variables": len(vars_jeu),
            "R² apprentissage": r2_score(app_t["prix"], modele_t.predict(app_t[vars_jeu])),
            "R² validation 2024": r2_score(val_t["prix"], modele_t.predict(val_t[vars_jeu])),
            "MAE validation 2024 (€/m²)": mean_absolute_error(val_t["prix"], modele_t.predict(val_t[vars_jeu])),
        })
display(pd.DataFrame(resultats).round(2))

In [ ]:
plt.figure(figsize=(22, 10))
for i, nom in enumerate(["Maisons", "Appartements"], start=1):
    modele_t, X_t = modeles[("sans redondances", nom)]
    echantillon_t = X_t.sample(min(2000, len(X_t)), random_state=42)
    valeurs_t = shap.LinearExplainer(modele_t, X_t).shap_values(echantillon_t)

    plt.subplot(1, 2, i)
    shap.summary_plot(valeurs_t, echantillon_t, max_display=30, show=False,
                      plot_size=None, color_bar=(i == 2))
    plt.title(f"SHAP – {nom}, sans redondances (régression linéaire, 2017-2023)")

plt.tight_layout()
plt.show()

**Comment lire le tableau (exemple : maisons, toutes les variables) :**
- **R² validation 2024 = 0,80** : le modèle explique 80 % des écarts de prix entre sous-bassins en 2024, une année qu'il n'a pas vue.
- **R² apprentissage = 0,80** : même valeur sur les années d'entraînement. Un R² nettement plus haut en apprentissage qu'en validation signalerait un modèle qui apprend par cœur ; ce n'est pas le cas.
- **MAE = 324 €/m²** : en 2024, le prix prévu d'un sous-bassin s'écarte en moyenne de 324 €/m² du prix réel.

**Constat :**
- **Le retrait des 3 variables redondantes ne change presque rien** : R² de validation identique (0,80 maisons, 0,86 appartements), erreur en hausse de 4 €/m² (maisons) et 3 €/m² (appartements), soit environ 1 %. Ces variables n'apportaient pas d'information propre : un modèle plus simple, aussi bon.
- **Pas de surapprentissage** : R² d'apprentissage et de validation presque égaux pour les deux types de bien.
- **Les variables du territoire seules expliquent déjà 80 à 86 % des écarts de prix entre sous-bassins**, sans le prix de l'année précédente. L'erreur reste toutefois élevée (324 et 476 €/m²) : ces variables disent où les prix sont hauts ou bas, pas comment ils évoluent d'une année à l'autre.

**SHAP sans redondances :**
- **Les variables principales ne changent pas** : niveau de vie en tête pour les deux types ; services, ventes de N-1 et densité pour les maisons ; sans voiture, densité, logements récents et petits logements pour les appartements.
- **Un signe à lire avec prudence** : seule, la part de ménages sans voiture est liée négativement au prix (rho = −0,25 et −0,18) ; dans le modèle, elle le fait monter. Ce n'est plus un effet de redondance : c'est l'effet *à niveau de vie et densité égaux*. Parmi des territoires de même revenu et de même densité, ceux où beaucoup de ménages vivent sans voiture (centres-villes) sont plus chers.
- **Des variables peu utilisées dans les deux modèles** : cambriolages, stupéfiants, équipements de commerces, part des indicateurs estimés, part des ménages couverts, vacance LOVAC. La vacance LOVAC apporte peu à côté de la vacance du recensement. Ce sont des candidates à tester en retrait à la modélisation, pas à retirer sur cette seule base (un arbre peut les utiliser autrement).
- **Ventes de N-1 et nombre de logements** (rho = 0,6) continuent de s'opposer pour les maisons : à surveiller.

### 5.4 Comparaison avec une forêt aléatoire

Forêt aléatoire (200 arbres, profondeur 10, au moins 20 lignes par feuille) par type de bien, mêmes variables (table en log, sans redondances), entraînée sur les prix de 2017 à 2023, mesurée sur 2024. Le log ne change rien pour la forêt (mêmes coupures) : il est gardé pour comparer avec la régression sur la même table.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

vars_rf = JEUX["sans redondances"]
resultats_rf, modeles_rf = [], {}
for nom, valeur in [("Maisons", 1), ("Appartements", 0)]:
    app_t = apprentissage[apprentissage["maison"] == valeur]
    val_t = validation[validation["maison"] == valeur]
    foret = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=20,
                                  n_jobs=-1, random_state=42).fit(app_t[vars_rf], app_t["prix"])
    modeles_rf[nom] = (foret, app_t[vars_rf])
    resultats_rf.append({
        "modèle": "forêt aléatoire", "type": nom,
        "R² apprentissage": r2_score(app_t["prix"], foret.predict(app_t[vars_rf])),
        "R² validation 2024": r2_score(val_t["prix"], foret.predict(val_t[vars_rf])),
        "MAE validation 2024 (€/m²)": mean_absolute_error(val_t["prix"], foret.predict(val_t[vars_rf])),
    })

# Comparaison avec la régression linéaire sans redondances
lineaire = (pd.DataFrame(resultats).query("variables == 'sans redondances'")
            .drop(columns=["variables", "nb variables"]).assign(**{"modèle": "régression linéaire"}))
display(pd.concat([lineaire, pd.DataFrame(resultats_rf)])
        .sort_values(["type", "modèle"]).round(2).reset_index(drop=True))

In [ ]:
plt.figure(figsize=(22, 10))
for i, nom in enumerate(["Maisons", "Appartements"], start=1):
    foret, X_t = modeles_rf[nom]
    echantillon_t = X_t.sample(min(2000, len(X_t)), random_state=42)
    valeurs_t = shap.TreeExplainer(foret).shap_values(echantillon_t)

    plt.subplot(1, 2, i)
    shap.summary_plot(valeurs_t, echantillon_t, max_display=30, show=False,
                      plot_size=None, color_bar=(i == 2))
    plt.title(f"SHAP – {nom}, forêt aléatoire (2017-2023)")

plt.tight_layout()
plt.show()

**Comment lire le tableau (exemple : maisons) :** la forêt aléatoire explique 88 % des écarts de prix en 2024 (R² validation), contre 80 % pour la régression linéaire, et son erreur moyenne baisse de 329 à 221 €/m² (−33 %). Pour les appartements, l'erreur passe de 476 à 350 €/m² (−27 %). R² d'apprentissage et de validation restent proches (0,87 / 0,88 pour les maisons ; 0,95 / 0,91 pour les appartements) : pas de surapprentissage marqué.

**Constat :**
- **La forêt prévoit nettement mieux que la régression** : une partie des liens avec le prix n'est pas en ligne droite (effets de seuil) ou dépend des autres variables (interactions). C'est ce que montre le mélange des couleurs dans son SHAP.
- **Les variables les plus utilisées changent d'un modèle à l'autre.** La régression donne l'effet de chaque variable, les autres égales ; la forêt choisit une variable par groupe de variables liées (logements anciens pour les maisons, vacance pour les appartements). L'importance dépend du modèle, pas seulement de la variable.
- **Effets de seuil visibles :** pour les maisons, peu de logements d'avant 1946 ajoutent de 500 à 1 600 €/m², beaucoup en retirent environ 300 ; quelques territoires très urbains (forte part d'appartements) ajoutent jusqu'à 4 200 €/m².
- **La densité sert d'étiquette « Paris » pour les appartements** : un paquet isolé à +7 000 / +8 000 €/m² correspond aux arrondissements, qui ont tous la densité de leur ville. Ce n'est pas un effet général de la densité.
- **Ce qui ressort des deux modèles** (conclusions solides) : niveau de vie, vacance, ancienneté d'emménagement, équipements de services et de sport, résidences secondaires parmi les variables les plus utilisées ; délinquance peu utile en dehors des vols sans violence ; variables de contrôle qualité et vacance LOVAC peu utilisées.
- Ce n'est pas une évaluation de modèle final : sans prix N-1, réglages non optimisés, années 2017-2023.

### 5.5 Variables qui ne comptent dans aucun modèle

Pour chacun des 4 modèles (régression et forêt, maisons et appartements) : part de chaque variable dans l'importance totale, à partir des valeurs SHAP (moyenne en valeur absolue). On garde la part la plus élevée des quatre : une variable sous 2 % dans tous les modèles ne compte nulle part. Seuil indicatif : avec 30 variables, une variable moyenne pèse environ 3 %.

In [ ]:
# Part de chaque variable dans l'importance SHAP totale de chaque modèle (en %)
parts = {}
for nom in ["Maisons", "Appartements"]:
    modele_lin, X_t = modeles[("sans redondances", nom)]
    foret, _ = modeles_rf[nom]
    echantillon_t = X_t.sample(min(2000, len(X_t)), random_state=42)
    for nom_modele, valeurs in [
        ("régression", shap.LinearExplainer(modele_lin, X_t).shap_values(echantillon_t)),
        ("forêt", shap.TreeExplainer(foret).shap_values(echantillon_t)),
    ]:
        imp = pd.Series(np.abs(valeurs).mean(axis=0), index=X_t.columns)
        parts[f"{nom} – {nom_modele}"] = imp / imp.sum() * 100

parts = pd.DataFrame(parts)
parts["part max (%)"] = parts.max(axis=1)
parts["compte nulle part (< 2 %)"] = parts["part max (%)"] < 2
display(parts.sort_values("part max (%)").round(1))

**Comment lire le tableau (exemple : `taux3_cambriolages`) :** dans chacun des 4 modèles, les cambriolages pèsent au plus 1,1 % de l'importance totale (régression des appartements). Avec 30 variables, une variable moyenne pèse environ 3 % : les cambriolages ne comptent dans aucun modèle.

**Constat :**
- **7 variables comptent pour moins de 2 % dans les 4 modèles** :
  - deux variables de contrôle qualité : `part_indicateurs_estimes` (0,6 % au plus), `part_menages_couverts` (1,2 %) ;
  - trois indicateurs de délinquance : stupéfiants (0,8 %), cambriolages (1,1 %), vols violents (1,9 %) ;
  - les équipements de santé (1,8 %) ;
  - la vacance LOVAC (2,0 % après arrondi, juste sous le seuil), qui n'apporte rien à côté de la vacance du recensement (jusqu'à 20 %).
- **Juste au-dessus du seuil** : violences sexuelles (2,6 %).
- **Des variables fortes dans un seul modèle**, à ne pas retirer : logements d'avant 1946 (28 % dans la forêt des maisons, 1 % dans la régression), ventes de N-1 (7 % dans la régression des maisons, presque rien ailleurs), ménages sans voiture (régressions seulement).
- **Les variables qui comptent partout** : niveau de vie, vacance du recensement, densité, équipements de services, ancienneté d'emménagement, résidences secondaires.
- **La délinquance apporte peu** : 3 des 8 indicateurs ne comptent nulle part. Seuls les vols sans violence (appartements) et les violences intrafamiliales dépassent 4 % dans un modèle.

# Synthèse : audit et EDA de la table de modélisation

| Constat majeur | Preuve | Conséquence projet | Action |
|---|---|---|---|
| La table compte une seule variable par information | Sections 1 et 2 : 28 450 lignes, 38 colonnes ; délinquance en taux lissés sur 3 ans, équipements pour 10 000 habitants, densité, part des ménages couverts ; un seul nombre restant (`nb_ventes_n1`) | Les territoires de tailles différentes sont comparables | Fait dans le feature engineering |
| Peu de valeurs vides, toutes expliquées | Délinquance vide pour les prix 2015-2016 (17,4 %) ; LOVAC vide pour un sous-bassin (11 lignes) | La régression linéaire refuse les cases vides | À la modélisation : retirer ces lignes (ou les compléter) pour les modèles qui l'exigent |
| 22 variables très asymétriques, avec des valeurs isolées réelles | Section 3 : asymétrie > 1 ; valeurs isolées = centres touristiques, Paris | Une régression linéaire serait tirée par quelques territoires | Log pour la régression, le modèle hiérarchique et le réseau de neurones ; inutile pour les arbres (tableau « À prendre en compte pour la modélisation ») |
| Trois couples de variables redondantes | Section 4 : petits logements / suroccupation (0,88), sans voiture / parking (−0,84), vols sans violence / destructions (0,78) ; SHAP de la régression : une variable gardée au hasard, ou les deux opposées | Retirer `part_suroccupes`, `part_parking`, `taux3_destructions` ne change ni le R² ni l'erreur (+1 %) | Retirer ces 3 variables avant la modélisation |
| Les variables du territoire expliquent l'essentiel du niveau des prix | Régression linéaire sans prix N-1, validée sur 2024 : R² 0,80 (maisons) et 0,86 (appartements), sans surapprentissage ; erreur de 329 et 476 €/m² | Elles disent où les prix sont hauts ou bas, pas comment ils évoluent | Le prix N-1 et les variables de marché restent indispensables pour l'évolution |
| La forêt aléatoire prévoit mieux que la régression linéaire | Validation 2024, sans prix N-1 : erreur de 221 contre 329 €/m² (maisons), 350 contre 476 €/m² (appartements) ; R² 0,88 et 0,91 contre 0,80 et 0,86 | Effets de seuil et interactions : une partie des liens avec le prix n'est pas en ligne droite | Comparer régression et modèles à arbres à la modélisation ; la progression du plus simple au plus complexe est justifiée |
| Les variables importantes dépendent du modèle et du type de bien ; un noyau commun | SHAP régression et forêt : niveau de vie, vacance, ancienneté d'emménagement, équipements de services et de sport, résidences secondaires en haut des deux. Variable de tête de la forêt : logements d'avant 1946 (maisons), vacance (appartements). La densité sert d'étiquette « Paris » pour la forêt | Confirme le choix de modéliser maisons et appartements séparément ; ne pas conclure sur un seul modèle ; densité des arrondissements = celle de leur ville | Densité par arrondissement à envisager (superficie IGN) |
| 6 variables ne comptent dans aucun modèle | Part de l'importance SHAP < 2 % dans les 4 modèles : `part_indicateurs_estimes`, `part_menages_couverts`, `taux3_stupefiants`, `taux3_cambriolages`, `taux3_vols_violents`, `taux_equip_sante` ; vacance LOVAC à 2,0 %. La variable de contrôle `part_logements_taux_impute` sert d'indice rural dans la régression des maisons (4,5 %) | Elles alourdissent les modèles sans les améliorer ; les variables de contrôle qualité mesurent la donnée, pas le territoire | Candidates au retrait, à confirmer à la modélisation (erreur avec et sans) ; variables de contrôle qualité : retrait à décider |
| 7 variables ne comptent dans aucun modèle | Part de l'importance SHAP < 2 % dans les 4 modèles : `part_indicateurs_estimes`, `part_menages_couverts`, `taux3_stupefiants`, `taux3_cambriolages`, `taux3_vols_violents`, `taux_equip_sante`, `taux_vacance_complete` (vacance LOVAC). La variable de contrôle `part_logements_taux_impute` sert d'indice rural dans la régression des maisons (4,5 %) | Elles alourdissent les modèles sans les améliorer ; les variables de contrôle qualité mesurent la donnée, pas le territoire ; la vacance LOVAC fait double emploi avec celle du recensement | Candidates au retrait, à confirmer à la modélisation (erreur avec et sans) ; variables de contrôle qualité : retrait à décider |